## Date transformation



#### 1. Rename columns:
#####   &emsp;- Strip spaces >> add "_";
#####   &emsp;- Replace to lowercase;
#####   &emsp;- Remove special symbols;

#### 2. Replace date format for date type columns

#### 3. Save as delta files to Silver Zone

In [0]:
# Check raw data - in Bronze Zone
from pyspark.sql.functions import from_utc_timestamp, date_format
from pyspark.sql.types import TimestampType

table_name = []

for i in dbutils.fs.ls('mnt/bronze/public/'):
    table_name.append(i.name.split('/')[0])    

for i in table_name:
    path = '/mnt/bronze/public/' + i + '/' + i +'.parquet'
    df = spark.read.format('parquet').load(path)
    #df = df.toDF(*[col.replace("-", "_") for col in df.columns])
    print(f"Table name: {path}")
    print(f"Columns: {df.columns} \n")
    

Table name: /mnt/bronze/public/parsed_dealers/parsed_dealers.parquet
Columns: ['company_name', 'address', 'city'] 

Table name: /mnt/bronze/public/parsed_dussbmw/parsed_dussbmw.parquet
Columns: ['location', 'title', 'subtitle', 'year_km_fuel', 'price', 'detail_url'] 

Table name: /mnt/bronze/public/parsed_ekris/parsed_ekris.parquet
Columns: ['Title', 'Subtitle', 'Year', 'Mileage', 'Fuel_Type', 'Price', 'Monthly_Price', 'Details_Link', 'Location'] 

Table name: /mnt/bronze/public/parsed_vanpoelgeest/parsed_vanpoelgeest.parquet
Columns: ['Vehicle_Name', 'Vehicle_Subtitle', 'Vehicle_Price', 'Vehicle_Tags', 'Business_Finance', 'Location'] 

Table name: /mnt/bronze/public/pcodes/pcodes.parquet
Columns: ['straat', 'huisnummer', 'huisletter', 'huistoevoeging', 'woonplaats', 'postcode', 'x', 'y', 'lon', 'lat', 'oppervlakte', 'gebruiksdoelen', 'bouwjaar', 'id'] 

Table name: /mnt/bronze/public/pcodes_gemeente/pcodes_gemeente.parquet
Columns: ['Geo_Point', 'Geo_Shape', 'Year', 'Provincie_code', 

In [0]:
# Strip spaces in column names and replace symbols to "_" , etc.
# Save as Delta to Silver Zone

from pyspark.sql.functions import from_utc_timestamp, date_format
from pyspark.sql.types import TimestampType

symbols_to_replace = ["-", '"', "/", ' ']

for i in table_name:
    path = '/mnt/bronze/public/' + i + '/' + i +'.parquet'
    df = spark.read.format('parquet').load(path)
    column = df.columns
    
    for col in column:
        new_col = col
        for symbol in symbols_to_replace:
            new_col = new_col.rstrip(" ").lstrip(" ").replace(symbol, "_")
        new_col = new_col.lower()
        df = df.withColumnRenamed(col, new_col)

        if "created_on" in col or "modified_on" in col or "date" in col:
            df = df.withColumn(
                col, 
                date_format(
                    from_utc_timestamp(df[col].cast(TimestampType()), "UTC"), 
                    "yyyy-MM-dd"
                )
                )
    
    output_path = '/mnt/silver/public/' + i + '/'
    df.write.format('delta').mode('overwrite').save(output_path)
    print(f"File: {path} >> saved as delta >> {output_path}")


File: /mnt/bronze/public/parsed_dealers/parsed_dealers.parquet >> saved as delta >> /mnt/silver/public/parsed_dealers/
File: /mnt/bronze/public/parsed_dussbmw/parsed_dussbmw.parquet >> saved as delta >> /mnt/silver/public/parsed_dussbmw/
File: /mnt/bronze/public/parsed_ekris/parsed_ekris.parquet >> saved as delta >> /mnt/silver/public/parsed_ekris/
File: /mnt/bronze/public/parsed_vanpoelgeest/parsed_vanpoelgeest.parquet >> saved as delta >> /mnt/silver/public/parsed_vanpoelgeest/
File: /mnt/bronze/public/pcodes/pcodes.parquet >> saved as delta >> /mnt/silver/public/pcodes/
File: /mnt/bronze/public/pcodes_gemeente/pcodes_gemeente.parquet >> saved as delta >> /mnt/silver/public/pcodes_gemeente/
File: /mnt/bronze/public/pcodes_provincie/pcodes_provincie.parquet >> saved as delta >> /mnt/silver/public/pcodes_provincie/
File: /mnt/bronze/public/vehicles_model/vehicles_model.parquet >> saved as delta >> /mnt/silver/public/vehicles_model/


In [0]:
# check new column names in Silver Zone
from pyspark.sql.functions import from_utc_timestamp, date_format
from pyspark.sql.types import TimestampType

table_name = []

for i in dbutils.fs.ls('mnt/silver/public/'):
    table_name.append(i.name.split('/')[0])    

for i in table_name:
    path = '/mnt/silver/public/' + i
    df = spark.read.format('delta').load(path)
    print(f"Delta table name: {path}")
    print(f"Columns: {df.columns} \n")

Delta table name: /mnt/silver/public/parsed_dealers
Columns: ['company_name', 'address', 'city'] 

Delta table name: /mnt/silver/public/parsed_dussbmw
Columns: ['location', 'title', 'subtitle', 'year_km_fuel', 'price', 'detail_url'] 

Delta table name: /mnt/silver/public/parsed_ekris
Columns: ['title', 'subtitle', 'year', 'mileage', 'fuel_type', 'price', 'monthly_price', 'details_link', 'location'] 

Delta table name: /mnt/silver/public/parsed_vanpoelgeest
Columns: ['vehicle_name', 'vehicle_subtitle', 'vehicle_price', 'vehicle_tags', 'business_finance', 'location'] 

Delta table name: /mnt/silver/public/pcodes
Columns: ['straat', 'huisnummer', 'huisletter', 'huistoevoeging', 'woonplaats', 'postcode', 'x', 'y', 'lon', 'lat', 'oppervlakte', 'gebruiksdoelen', 'bouwjaar', 'id'] 

Delta table name: /mnt/silver/public/pcodes_gemeente
Columns: ['geo_point', 'geo_shape', 'year', 'provincie_code', 'provincie_name', 'gemeente_code', 'gemeente_name', 'type', 'gemeente_code_with_prefix'] 

Delta t

#####If we need to replace default 7-days VACUUM retention period to 2 days (48 h)

> spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

> from delta.tables import DeltaTable

> DeltaTable.forPath(spark, output_path).vacuum(retentionHours=48)